In [9]:
"""
Compute average frame-count, FPS, and duration by reading only the first
`N` lines from a local CSV file.

Dependencies (install once):
    pip install pandas boto3 opencv-python-headless tqdm
"""

import os
import tempfile
import boto3
import cv2
import pandas as pd
import numpy as np
from urllib.parse import urlparse
from tqdm.auto import tqdm

# --- Configuration --------------------------------------------------------
N = 1000  # ← How many videos to sample from the top of the CSV
CSV_FILE_PATH = "s3_mp4_files_filtered.csv"
# --------------------------------------------------------------------------

tmp_dir = tempfile.mkdtemp()
s3 = boto3.client("s3")

frames, fpss, durs = [], [], []

def get_videos_from_csv(path: str, num_rows: int) -> list[str]:
    """
    Reads the first `num_rows` of S3 URIs from a single-column CSV file.
    """
    try:
        print(f"Reading the first {num_rows} lines from {path}...")
        # Use `nrows` to read only the top N rows from the file
        df = pd.read_csv(path, header=None, nrows=num_rows)
        video_list = df[0].tolist()
        print(f"Found {len(video_list)} video URIs.")
        return video_list
    except FileNotFoundError:
        print(f"FATAL: The file '{path}' was not found.")
        return []
    except Exception as e:
        print(f"An error occurred while reading the CSV: {e}")
        return []

def download(uri: str, dest_dir: str) -> str:
    """Download S3 object to `dest_dir`, return local path."""
    parsed = urlparse(uri)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    local = os.path.join(dest_dir, os.path.basename(key))
    if os.path.exists(local):  # cached
        return local
    s3.download_file(bucket, key, local)
    return local


# Get only the first N videos from the local CSV file
video_list = get_videos_from_csv(CSV_FILE_PATH, N)

# The list length is already capped at N, so we can iterate directly
for uri in tqdm(video_list, total=len(video_list), desc="Processing Videos", unit="video"):
    try:
        path = download(uri, tmp_dir)
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():  # skip broken files
            print(f"Warning: Could not open video {os.path.basename(path)}")
            continue

        f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        r = cap.get(cv2.CAP_PROP_FPS) or np.nan
        cap.release()

        frames.append(f)
        fpss.append(r)
        durs.append(f / r if r and r > 0 else np.nan)

    except Exception as e:
        print(f"Error processing {uri}: {e}")
        continue

# --- Report ---------------------------------------------------------------
if frames:
    print(f"\nSampled videos : {len(frames)}")
    print(f"Avg Frames     : {np.nanmean(frames):.1f}")
    print(f"Avg FPS        : {np.nanmean(fpss):.2f}")
    print(f"Avg Duration (s): {np.nanmean(durs):.2f}")
else:
    print("\nNo videos were processed.")

Reading the first 1000 lines from s3_mp4_files_filtered.csv...
Found 1000 video URIs.


Processing Videos:   0%|          | 0/1000 [00:00<?, ?video/s]


Sampled videos : 1000
Avg Frames     : 48.3
Avg FPS        : 30.00
Avg Duration (s): 1.61
